# Rec occupancy: reproducible analysis
Run from the project directory. Live data only; synthetic tests stay separate. Read README.md before interpreting results. No historical results are fabricated.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
root = Path.cwd()
if not (root / 'poll_rec_counts.py').exists(): root = root.parent
sys.path.insert(0, str(root))
from poll_rec_counts import connect
from analyze import run, hourly_bins, constraint_bounds, level_profile
status = run(root)
status

## Data quality and available observations
Inspect age, resets, closures, revisions and opening-hours coverage before analyzing capacity.

In [ ]:
db = connect(root)
quality = pd.read_sql_query('SELECT * FROM quality_counts', db)
clean = pd.read_sql_query('SELECT * FROM clean_counts', db)
quality.groupby('facility').agg(readings=('location_id','size'), median_age_min=('age_min','median'), resets=('is_reset','sum'), verified_open=('verified_open','sum'))

## Profile and cross-room slack
Use observed hourly means; missing cells remain unknown. Recurring profiles require at least four weeks. The comparison below is descriptive and asynchronous.

In [ ]:
h = hourly_bins(clean)
if not h.empty:
    weights = h[h.location.str.contains('Strength|Conditioning', case=False)]
    profile = weights.groupby(['location','weekday','hour']).agg(utilization=('utilization','mean'), people=('people','mean'), capacity=('capacity','mean'), observed_hours=('hour_bin','size'))
    profile['slack_people'] = profile.capacity - profile.people
    display(profile)
else:
    print('Pending verified hours and readings')

## Threshold exposure sensitivity
Compare 60, 90 and 120 minute freshness caps. A unique bottleneck requires adequate comparable coverage; overlapping bounds leave the ranking uncertain.

In [ ]:
for cap in [60,90,120]:
    print('Freshness cap:',cap)
    display(constraint_bounds(db,quality,max_age=cap))

## Forecast evaluation
The analysis script enforces at least six weeks and a final two-week holdout. Metrics share the same evaluation rows. Calendar and lag features are implemented; weather vintages and sourced game dates are availability-gated.

In [ ]:
for f in sorted((root/'reports').glob('forecast_metrics_*.csv')):
    display(pd.read_csv(f))
print(status['forecast_status'])

## Control chart and January decay
Review Phase I normal days and daily coverage before running control.py with explicit dates. Do not select the baseline to conceal a shift. Limits and fits remain pending until enough real data exist.

In [ ]:
print(status['control_status'])
for f in sorted((root/'reports').glob('decay_*.json')):
    print(f.name, json.loads(f.read_text()))

## Occupancy leveling
This conserves occupancy person-hours. It is not an arrival-shift causal model. Use only a complete, adequately sampled profile.

In [ ]:
path=root/'reports/leveling.csv'
if path.exists() and path.stat().st_size>2:
    display(pd.read_csv(path))
else:
    print('Pending four observed dates per included weekday/hour bin')
db.close()